# Step 2: Informal read (stage A)

This notebook picks **42 traces** for you to read and describe in your own words. Your descriptions are the raw material for the first taxonomy (v0).

- **30 wrong-answer traces** (5 per model per dataset): each one is guaranteed to contain a failure.
- **12 correct-answer traces** (2 per model per dataset): to look for *hidden* errors, where the answer is right but the reasoning is flawed. Prior work found these in 14–24% of correct answers in commonsense QA (ReTraceQA), but only ~3.5% in GSM8K (ProcessBench).
- **21 math (GSM8K) + 21 commonsense (CommonsenseQA)**, all different questions, with correct and wrong traces mixed together.
- **Only traces that followed the format** (`format_ok`).
- The model name is **hidden** in the reading sheet, so it doesn't influence your descriptions.

**Output** (in `annotation/stage_a/`):
- `stage_a_reading_sheet.xlsx`: the 30 traces plus empty columns for your notes. **This is the file you fill in.**
- `stage_a_selection.csv`: which question and model each row came from. These questions will be kept out of the later annotation set.

No GPU needed. Run it from the repository folder (locally, or on Kaggle/Colab after cloning the repo).

## 1. Settings

In [ ]:
SEED = 2026              # fixed, so the same traces are picked every time
WRONG_PER_MODEL = 5      # wrong-answer traces per model per dataset   -> 30 in total
CORRECT_PER_MODEL = 2    # correct-answer traces per model per dataset -> 12 in total
TRACE_DIR = "data/traces"
OUT_DIR = "annotation/stage_a"

## 2. Load the traces

In [ ]:
import glob, json, os, random
import pandas as pd

rows = []
for path in sorted(glob.glob(f"{TRACE_DIR}/traces_*_300.jsonl")):
    with open(path, encoding="utf-8") as f:
        rows += [json.loads(line) for line in f if line.strip()]
traces = pd.DataFrame(rows)
print(len(traces), "traces from", traces["model"].nunique(), "models")

## 3. Pick the traces

For each dataset and model: 5 wrong-answer and 2 correct-answer traces, drawn at random with `SEED`. A question is never used twice, so all 21 questions per dataset are different.

In [ ]:
rng = random.Random(SEED)
usable = traces[traces["format_ok"]]

picked = []
for dataset in ["gsm8k", "csqa"]:
    used_questions = set()
    for model in sorted(usable["model"].unique()):
        for answer_correct, n in [(False, WRONG_PER_MODEL), (True, CORRECT_PER_MODEL)]:
            pool = usable[(usable["dataset"] == dataset) & (usable["model"] == model)
                          & (usable["correct"] == answer_correct)]
            pool = pool[~pool["question_id"].isin(used_questions)]
            chosen = pool.loc[sorted(rng.sample(list(pool.index), n))]
            used_questions |= set(chosen["question_id"])
            picked.append(chosen)

selection = pd.concat(picked)
# shuffle the order within each dataset, so models are mixed while reading
selection = pd.concat(
    g.sample(frac=1, random_state=SEED) for _, g in selection.groupby("dataset", sort=False)
).reset_index(drop=True)
selection["id"] = [f"A{i:02d}" for i in range(1, len(selection) + 1)]
print(selection.groupby(["dataset", "model", "correct"]).size().unstack().rename(columns={False: "wrong", True: "correct"}))

## 4. Read the traces here (optional)

The same content is in the Excel sheet. Use whichever is more comfortable.

In [ ]:
def show(t):
    print("=" * 90)
    print(f"{t['id']}  [{t['dataset']}]   correct answer: {t['gold_answer']}   model answer: {t['model_answer']}"
          f"   ({'RIGHT' if t['correct'] else 'WRONG'})")
    print("-" * 90)
    print(t["question"])
    if isinstance(t["options"], dict):
        for label, text in t["options"].items():
            print(f"   {label}. {text}")
    print("-" * 90)
    for i, step in enumerate(t["steps"], start=1):
        print(f"[{i}] {step}")

for _, t in selection.iterrows():
    show(t)

## 5. Save the reading sheet

The Excel file has two tabs:
- **instructions:** what to write in each column, with an example.
- **traces:** one row per trace. The yellow columns are yours to fill in.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

os.makedirs(OUT_DIR, exist_ok=True)

def options_text(o):
    return "\n".join(f"{k}. {v}" for k, v in o.items()) if isinstance(o, dict) else ""

def steps_text(steps):
    return "\n".join(f"[{i}] {s}" for i, s in enumerate(steps, start=1))

INFO = ["id", "task", "question", "options", "correct_answer", "model_answer", "answer_is_right", "steps"]
NOTES = ["reasoning_valid", "first_error_step", "description", "confidence", "note"]

wb = Workbook()
ins = wb.active
ins.title = "instructions"
for line in [
    ["Stage A: informal read. For each trace, read the question and the steps, then fill in the yellow columns."],
    [],
    ["Column", "What to write"],
    ["reasoning_valid", "yes / no. 'yes' only if every step is right. Wrong answers are almost always 'no'. Right answers can still be 'no' (a hidden error): check every step."],
    ["first_error_step", "The number [k] of the first step that is wrong. If only the final answer line is wrong, write 'final'. Leave empty if reasoning_valid = yes."],
    ["description", "One sentence in your own words: what went wrong in that step. Don't use category names yet. Leave empty if reasoning_valid = yes."],
    ["confidence", "1 = unsure, 2 = fairly sure, 3 = clear"],
    ["note", "Optional: a later error that is independent of the first one, or anything unusual."],
    [],
    ["Example", "Q: Tom buys 4 pens at $3 each and pays with $20. Change? Steps: [1] Each pen costs $3. [2] 4 pens cost 3 + 4 = 7. [3] 20 - 7 = 13."],
    ["", "reasoning_valid = no | first_error_step = 2 | description = added the price and the number of pens instead of multiplying | confidence = 3 | note = step 3 is right given step 2"],
]:
    ins.append(line)
ins["A1"].font = Font(bold=True)
ins.column_dimensions["A"].width = 20
ins.column_dimensions["B"].width = 120

ws = wb.create_sheet("traces")
ws.append(INFO + NOTES)
for _, t in selection.iterrows():
    ws.append([t["id"], "math" if t["dataset"] == "gsm8k" else "commonsense", t["question"],
               options_text(t["options"]), t["gold_answer"], t["model_answer"],
               "yes" if t["correct"] else "no", steps_text(t["steps"])]
              + [""] * len(NOTES))

widths = {"id": 6, "task": 12, "question": 45, "options": 25, "correct_answer": 10, "model_answer": 10, "answer_is_right": 10,
          "steps": 80, "reasoning_valid": 10, "first_error_step": 10, "description": 50, "confidence": 10, "note": 35}
yellow = PatternFill("solid", fgColor="FFF2CC")
for j, name in enumerate(INFO + NOTES, start=1):
    col = ws.cell(row=1, column=j).column_letter
    ws.column_dimensions[col].width = widths[name]
    ws.cell(row=1, column=j).font = Font(bold=True)
    if name in NOTES:
        for r in range(1, len(selection) + 2):
            ws.cell(row=r, column=j).fill = yellow
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
ws.freeze_panes = "C2"

last = len(selection) + 1
for name, values in [("reasoning_valid", '"yes,no"'), ("confidence", '"1,2,3"')]:
    column = ws.cell(row=1, column=(INFO + NOTES).index(name) + 1).column_letter
    dv = DataValidation(type="list", formula1=values, allow_blank=True)
    dv.add(f"{column}2:{column}{last}")
    ws.add_data_validation(dv)

wb.save(f"{OUT_DIR}/stage_a_reading_sheet.xlsx")
selection[["id", "dataset", "question_id", "model", "gold_answer", "model_answer", "correct"]].to_csv(
    f"{OUT_DIR}/stage_a_selection.csv", index=False)
print("saved", f"{OUT_DIR}/stage_a_reading_sheet.xlsx", "and", f"{OUT_DIR}/stage_a_selection.csv")

## What happens next

1. Fill in the yellow columns for all 42 traces, then upload the sheet back to `annotation/stage_a/`.
2. Group similar descriptions into categories. That's taxonomy v0.
3. Compare your groups with ReTraceQA's types (misinterpretation, hallucination, reasoning error), and note which appear only in math, only in commonsense, or in both.